# Train individual uplift models

## tl;dr

AdLift fits separate S-, T-, and pooled X-style uplift learners for Visit and Conversion. Training and early stopping use only the train and validation partitions. Frozen models then score test features without reading test outcomes.

The saved diagnostics confirm that all three learners produce finite, non-constant uplift scores. Ordinary response AUC, log loss, and Brier score are training diagnostics; they do not determine which uplift ranking is best. Ranking quality and the final policy decision are evaluated in the next notebook.

## Context & Methods

For pre-treatment features $X$, randomized assignment $T$, and binary outcome $Y$, the target score is:

$$
\tau(x)=P(Y=1\mid X=x,T=1)-P(Y=1\mid X=x,T=0).
$$

| Learner | Implementation |
|---|---|
| S-Learner | Fit one response model $\hat\mu(x,t)$ and score $\hat\mu(x,1)-\hat\mu(x,0)$. |
| T-Learner | Fit separate treatment and control response models and score $\hat\mu_1(x)-\hat\mu_0(x)$. |
| Pooled X-style | Impute effect pseudo-outcomes from the T-Learner response models, pool both arms, and fit one effect regression. |

The artifact called `x_learner` is a pooled single-effect-model variant. It is not the canonical two-effect-regressor, propensity-blended X-Learner.

### Key assumptions and boundaries

Only `f0`–`f11` enter model matrices. `exposure`, outcomes, row identifiers, and split labels are prohibited predictive features. `treatment` is used only where the learner definition requires the intervention indicator. Test outcomes remain null in Phase 3 score artifacts and are joined only during the frozen Phase 4 evaluation.

## Setup

This reader-facing notebook uses checked-in configuration and diagnostic tables. Model fitting remains in `src/adlift/models.py` and is run through `scripts/train_models.py`. It does not retrain 13.98 million records while being viewed.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
TABLES = PROJECT_ROOT / "results" / "tables"
CONFIG = json.loads((PROJECT_ROOT / "config" / "models.json").read_text())
EXPECTED_MODELS = {"s_learner", "t_learner", "x_learner"}

## Model specification

The configuration below records the allowed features, development seed, outcomes, and main LightGBM complexity controls. Visit and Conversion use separate settings because Conversion is much rarer.

In [2]:
specification = pd.DataFrame({
    "item": ["features", "outcomes", "development seed", "visit leaves / depth", "conversion leaves / depth"],
    "value": [
        ", ".join(CONFIG["features"]),
        ", ".join(CONFIG["outcomes"]),
        str(CONFIG["seed"]),
        f'{CONFIG["lightgbm"]["visit_binary"]["num_leaves"]} / {CONFIG["lightgbm"]["visit_binary"]["max_depth"]}',
        f'{CONFIG["lightgbm"]["conversion_binary"]["num_leaves"]} / {CONFIG["lightgbm"]["conversion_binary"]["max_depth"]}',
    ],
})
display(specification)

,item,value
0,features,"f0, f1, f2, f3, f4, f5, f6, f7, f8, f9, f10, f11"
1,outcomes,"visit, conversion"
2,development seed,20260816
3,visit leaves / depth,31 / 8
4,conversion leaves / depth,15 / 6


## Validation diagnostics and score distributions

Response diagnostics check whether the underlying outcome models trained sensibly. The uplift-score rows verify that every learner produces a continuous treatment-minus-control score. These quantities are not individual causal ground truth and are not used to select the final learner.

In [3]:
diagnostic_frames = []
for outcome in CONFIG["outcomes"]:
    frame = pd.read_csv(TABLES / f"phase3/full_{outcome}_model_diagnostics.csv")
    frame.insert(0, "outcome", outcome)
    diagnostic_frames.append(frame)

diagnostics = pd.concat(diagnostic_frames, ignore_index=True)
response_diagnostics = diagnostics[diagnostics["arm"].isin(["0", "1", 0, 1])][
    ["outcome", "model_name", "arm", "log_loss", "brier_score", "roc_auc", "pr_auc"]
]
uplift_scores = diagnostics[diagnostics["arm"] == "uplift_score"][[
    "outcome", "model_name", "score_mean", "score_standard_deviation", "score_minimum", "score_maximum"
]]

display(response_diagnostics.round(6))
display(uplift_scores.round(6))

,outcome,model_name,arm,log_loss,brier_score,roc_auc,pr_auc
0,visit,s_learner,0,0.089108,0.025328,0.947748,0.481818
1,visit,s_learner,1,0.105147,0.030464,0.946481,0.524810
2,visit,t_learner,0,0.089347,0.025397,0.947440,0.478769
3,visit,t_learner,1,0.105148,0.030463,0.946460,0.524782
7,conversion,s_learner,0,0.008029,0.001641,0.961875,0.233835
8,conversion,s_learner,1,0.012165,0.002607,0.958721,0.243169
9,conversion,t_learner,0,0.008041,0.001647,0.962132,0.228252
10,conversion,t_learner,1,0.012153,0.002604,0.958564,0.244097


,outcome,model_name,score_mean,score_standard_deviation,score_minimum,score_maximum
4,visit,s_learner,0.007271,0.022770,-0.031102,0.281831
5,visit,t_learner,0.007378,0.025836,-0.208208,0.408999
6,visit,x_learner,0.007379,0.023466,-0.054786,0.334013
11,conversion,s_learner,0.000926,0.005098,-0.000004,0.134396
12,conversion,t_learner,0.000991,0.006366,-0.134592,0.301978
13,conversion,x_learner,0.000987,0.005013,-0.014091,0.078531


The treatment and control response components have similar validation diagnostics within each outcome. All three learners generate heterogeneous scores, including some negative predictions. Scores are retained without clipping because ranking must preserve predicted harm as well as predicted benefit.

The average predicted score is not treated as the randomized ATE, and a high response AUC is not evidence of good uplift ranking. Both questions require held-out treatment-control comparisons.

## Stability check

One alternate training seed provides a limited sensitivity check. Spearman correlation measures full-ranking agreement; top-set overlap measures agreement among the highest-ranked 10%. One alternate seed cannot establish production stability.

In [4]:
stability = pd.concat(
    [pd.read_csv(TABLES / f"phase4/{outcome}_phase3_stability.csv") for outcome in CONFIG["outcomes"]],
    ignore_index=True,
)
display(stability[[
    "outcome_name", "model_name", "reference_seed", "comparison_seed",
    "spearman_rank_correlation", "top_fraction", "top_set_overlap"
]].round(4))

,outcome_name,model_name,reference_seed,comparison_seed,spearman_rank_correlation,top_fraction,top_set_overlap
0,visit,s_learner,20260816,20260817,0.9784,0.1,0.9400
1,visit,t_learner,20260816,20260817,0.8539,0.1,0.8849
2,visit,x_learner,20260816,20260817,0.9778,0.1,0.9649
3,conversion,s_learner,20260816,20260817,0.9510,0.1,0.9545
4,conversion,t_learner,20260816,20260817,0.8758,0.1,0.8955
5,conversion,x_learner,20260816,20260817,0.9530,0.1,0.9383


## Checks

In [5]:
assert set(uplift_scores["model_name"]) == EXPECTED_MODELS
assert set(uplift_scores["outcome"]) == set(CONFIG["outcomes"])
assert uplift_scores[["score_mean", "score_standard_deviation", "score_minimum", "score_maximum"]].notna().all().all()
assert (uplift_scores["score_standard_deviation"] > 0).all()
assert set(stability["model_name"]) == EXPECTED_MODELS
assert stability["spearman_rank_correlation"].between(-1, 1).all()
assert stability["top_set_overlap"].between(0, 1).all()
print("Saved model diagnostics and stability tables passed the notebook checks.")

Saved model diagnostics and stability tables passed the notebook checks.


## Takeaways

- S-, T-, and pooled X-style learners were trained separately for Visit and Conversion and produced valid continuous uplift scores.
- Test outcomes were excluded from model fitting and score creation.
- Response diagnostics support engineering QA but cannot choose an uplift learner.
- The single alternate-seed check is informative but limited.
- Continue to `04_evaluate_targeting.ipynb` to compare cumulative gain, Qini, paired uncertainty, and budget-specific policy value.